# 📊 Thesis — All-Model Comparison Notebook

**EV Charging Load Demand Prediction — 5-Fold Cross-Validation**

This notebook consolidates **exact metric values** from four separate model notebooks:

| Notebook | Models |
|----------|--------|
| `Classical_ML.ipynb` | Linear Regression, Decision Tree, Random Forest, XGBoost |
| `ANN_KFold_v2.ipynb` | ANN (KFold) |
| `LSTM_K_Fold_.ipynb` | LSTM (KFold) |
| `GRU_K_Fold_.ipynb`  | GRU (KFold) |

**No data is changed** — all numbers are taken verbatim from each notebook's outputs.

---
### Contents
- **Cell 1** — Imports
- **Cell 2** — Hardcoded metric data (from notebook outputs)
- **Cell 3** — Regression Metrics Table (MAE, MSE, RMSE, MAPE, R²)
- **Cell 4** — Regression Metrics Bar Chart
- **Cell 5** — Accuracy & F1-Score Table
- **Cell 6** — Accuracy & F1-Score Bar Charts
- **Cell 7** — Per-Class F1 Grouped Bar Chart
- **Cell 8** — Overall Score Comparison Radar Chart
- **Cell 9** — Combined Summary Heatmap
- **Cell 10** — Download All Figures

## Cell 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker

print('All imports done ✓')

## Cell 2 — Hardcoded Metric Data

All values taken **verbatim** from each notebook’s printed outputs (no rounding or changes).

| Source | Cell with output |
|--------|------------------|
| Classical ML (LR, DT, RF, XGB) | `Classical_ML.ipynb` Cell 10 + Cell 16 |
| ANN | `ANN_KFold_v2.ipynb` Cell 10 |
| LSTM | `LSTM_K_Fold_.ipynb` Cell 22 |
| GRU  | `GRU_K_Fold_.ipynb` Cell 15 |

In [ ]:
# ============================================================
# REGRESSION METRICS  (source: each notebook's printed output)
# ============================================================
reg_data = [
    # Model               MAE        MSE          RMSE       MAPE(%)    R2
    # --- Classical ML (Classical_ML.ipynb  Cell 10) ---
    ('Linear Regression', 25.702,    1068.805,    32.693,    5.465,     0.9946),
    ('Decision Tree',     33.530,    1896.711,    43.551,    7.389,     0.9904),
    ('Random Forest',     39.705,    2397.463,    48.964,    7.199,     0.9878),
    ('XGBoost',           41.712,    2864.295,    53.519,    6.535,     0.9854),
    # --- ANN (ANN_KFold_v2.ipynb  Cell 10) ---
    ('ANN',               20.232,    687.610,     26.222,    4.5676,    0.9965),
    # --- LSTM (LSTM_K_Fold_.ipynb  Cell 22) ---
    ('LSTM',              90.008,    10027.024,   100.135,   9.801,     0.9308),
    # --- GRU (GRU_K_Fold_.ipynb  Cell 15) ---
    ('GRU',               62.090,    5863.163,    70.346,    7.167,     0.9593),
]

reg_df = pd.DataFrame(reg_data,
    columns=['Model', 'MAE', 'MSE', 'RMSE', 'MAPE (%)', 'R²'])

# ============================================================
# CLASSIFICATION METRICS  (Accuracy & F1)
# ============================================================
cls_data = [
    # Model               Acc(%)      F1_wtd      F1_Low     F1_Med     F1_High    F1_Peak
    # --- Classical ML (Classical_ML.ipynb  Cell 16) ---
    ('Linear Regression', 94.5349,    0.8672,     0.8066,    0.7697,    0.8968,    1.0000),
    ('Decision Tree',     92.6109,    0.8637,     0.9773,    0.8908,    0.7607,    0.8197),
    ('Random Forest',     92.8006,    0.8506,     0.8974,    0.8489,    0.8014,    0.8552),
    ('XGBoost',           93.4646,    0.8475,     0.8418,    0.7782,    0.8837,    0.8789),
    # --- ANN (ANN_KFold_v2.ipynb  Cell 10) ---
    ('ANN',               95.4324,    0.9218,     0.9762,    0.8984,    0.8850,    0.9219),
    # --- LSTM (LSTM_K_Fold_.ipynb  Cell 22) ---
    ('LSTM',              90.1989,    0.5328,     0.9076,    0.7934,    0.5190,    0.0000),
    # --- GRU (GRU_K_Fold_.ipynb  Cell 15) ---
    ('GRU',               92.8335,    0.6724,     0.9364,    0.8641,    0.6423,    0.2976),
]

cls_df = pd.DataFrame(cls_data,
    columns=['Model', 'Accuracy (%)', 'F1 (weighted)',
             'F1_Low', 'F1_Medium', 'F1_High', 'F1_Peak'])

# Colour palette — one per model, consistent across all plots
MODEL_COLORS = {
    'Linear Regression': '#2196F3',   # blue
    'Decision Tree':     '#009688',   # teal
    'Random Forest':     '#FF9800',   # orange
    'XGBoost':           '#F44336',   # red
    'ANN':               '#9C27B0',   # purple
    'LSTM':              '#00BCD4',   # cyan
    'GRU':               '#4CAF50',   # green
}

MODEL_ORDER   = list(MODEL_COLORS.keys())
BAR_COLORS    = [MODEL_COLORS[m] for m in MODEL_ORDER]

print('Data loaded ✓')
print(f'  Regression rows  : {len(reg_df)}')
print(f'  Classification rows: {len(cls_df)}')

## Cell 3 — Regression Metrics Table (MAE, MSE, RMSE, MAPE, R²)

Formula reference:
- **MAE** = mean|y − ŷ|  
- **MSE** = mean(y − ŷ)²  
- **RMSE** = √MSE  
- **MAPE** = mean|y−ŷ|/y × 100  (non-zero samples)  
- **R²** = 1 − SS_res/SS_tot

In [ ]:
# ── Pretty-print regression table ─────────────────────────────────────────────────────────────────────────
W = 82
print('=' * W)
print('  REGRESSION METRICS — All Models  (5-Fold CV)')
print('  Source: Classical_ML.ipynb | ANN_KFold_v2.ipynb | LSTM_K_Fold_.ipynb | GRU_K_Fold_.ipynb')
print('=' * W)
hdr = f"{'Model':<22} {'MAE':>9} {'MSE':>12} {'RMSE':>9} {'MAPE(%)':>9} {'R²':>8}"
print(hdr)
print('─' * W)

for _, r in reg_df.iterrows():
    print(f"{r['Model']:<22} {r['MAE']:>9.3f} {r['MSE']:>12.3f} "
          f"{r['RMSE']:>9.3f} {r['MAPE (%)']:>9.4f} {r['R²']:>8.4f}")

print('=' * W)
print('Best MAE  →', reg_df.loc[reg_df['MAE'].idxmin(),      'Model'])
print('Best MSE  →', reg_df.loc[reg_df['MSE'].idxmin(),      'Model'])
print('Best RMSE →', reg_df.loc[reg_df['RMSE'].idxmin(),     'Model'])
print('Best MAPE →', reg_df.loc[reg_df['MAPE (%)'].idxmin(), 'Model'])
print('Best R²   →', reg_df.loc[reg_df['R²'].idxmax(),       'Model'])

## Cell 4 — Regression Metrics Bar Chart Visualization

In [ ]:
metrics_cfg = [
    ('MAE',      'MAE (kW)',           False),
    ('MSE',      'MSE (kW²)',          False),
    ('RMSE',     'RMSE (kW)',           False),
    ('MAPE (%)', 'MAPE (%)',            False),
    ('R²',       'R² (higher = better)', True ),
]

fig, axes = plt.subplots(1, 5, figsize=(26, 6))
fig.suptitle(
    'Regression Metrics Comparison — All Models (5-Fold CV)',
    fontsize=14, fontweight='bold', y=1.03
)

models = reg_df['Model'].tolist()
colors = [MODEL_COLORS[m] for m in models]
x = np.arange(len(models))

for ax, (col, ylabel, higher_better) in zip(axes, metrics_cfg):
    vals = reg_df[col].values
    bars = ax.bar(x, vals, color=colors, edgecolor='white', linewidth=0.6, width=0.6)
    best_idx = int(np.argmax(vals)) if higher_better else int(np.argmin(vals))
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2.2)

    for bar, val in zip(bars, vals):
        fmt = f'{val:.4f}' if col == 'R²' else (f'{val:.2f}' if col == 'MAPE (%)' else f'{val:.1f}')
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.01,
                fmt, ha='center', va='bottom', fontsize=6.5, fontweight='bold')

    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=7)
    ax.set_xlabel('↑ better' if higher_better else '↓ better', fontsize=7.5, color='dimgray')
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('comparison_regression_metrics.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved → comparison_regression_metrics.png')

## Cell 5 — Accuracy & F1-Score Table

- **Accuracy (%)** = 100 − MAPE  (non-zero y samples)
- **F1 (weighted)** = harmonic mean of Precision & Recall, weighted by class support
- **Per-class F1**: Low (0–280 kW) / Medium (280–830 kW) / High (830–1165 kW) / Peak (1165+ kW)

In [ ]:
W = 82
print('=' * W)
print('  ACCURACY & F1-SCORE TABLE — All Models  (5-Fold CV)')
print('  Accuracy (%) = 100 − MAPE  [non-zero samples only]')
print('=' * W)
hdr = (f"{'Model':<22} {'Acc (%)':>10} {'F1-Wtd':>9} "
       f"{'F1-Low':>8} {'F1-Med':>8} {'F1-High':>8} {'F1-Peak':>8}")
print(hdr)
print('─' * W)
for _, r in cls_df.iterrows():
    print(f"{r['Model']:<22} {r['Accuracy (%)']:>10.4f} {r['F1 (weighted)']:>9.4f} "
          f"{r['F1_Low']:>8.4f} {r['F1_Medium']:>8.4f} "
          f"{r['F1_High']:>8.4f} {r['F1_Peak']:>8.4f}")
print('=' * W)
print('Best Accuracy     →', cls_df.loc[cls_df['Accuracy (%)'].idxmax(),    'Model'])
print('Best F1 (weighted)→', cls_df.loc[cls_df['F1 (weighted)'].idxmax(),   'Model'])
print('Best F1 Low       →', cls_df.loc[cls_df['F1_Low'].idxmax(),          'Model'])
print('Best F1 Medium    →', cls_df.loc[cls_df['F1_Medium'].idxmax(),       'Model'])
print('Best F1 High      →', cls_df.loc[cls_df['F1_High'].idxmax(),         'Model'])
print('Best F1 Peak      →', cls_df.loc[cls_df['F1_Peak'].idxmax(),         'Model'])

## Cell 6 — Accuracy & Weighted F1-Score Bar Charts

In [ ]:
models_cls = cls_df['Model'].tolist()
colors_cls = [MODEL_COLORS[m] for m in models_cls]
x = np.arange(len(models_cls))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    'Accuracy & F1-Score Comparison — All Models (5-Fold CV)',
    fontsize=13, fontweight='bold'
)

# ── Accuracy bar ─────────────────────────────────────────────────────────────────────────
ax = axes[0]
vals = cls_df['Accuracy (%)'].values
bars = ax.bar(x, vals, color=colors_cls, edgecolor='white', linewidth=0.6, width=0.6)
best = int(np.argmax(vals))
bars[best].set_edgecolor('black'); bars[best].set_linewidth(2.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.06,
            f'{val:.4f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_title('Accuracy (%) = 100 − MAPE', fontsize=11, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([m.replace(' ', '\n') for m in models_cls], fontsize=8.5)
ax.set_ylim(85, 100)
ax.set_xlabel('↑ higher = better', fontsize=8, color='dimgray')
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.spines[['top', 'right']].set_visible(False)

# ── F1 Weighted bar ───────────────────────────────────────────────────────────────────
ax = axes[1]
vals = cls_df['F1 (weighted)'].values
bars = ax.bar(x, vals, color=colors_cls, edgecolor='white', linewidth=0.6, width=0.6)
best = int(np.argmax(vals))
bars[best].set_edgecolor('black'); bars[best].set_linewidth(2.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_title('F1-Score (Weighted)', fontsize=11, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([m.replace(' ', '\n') for m in models_cls], fontsize=8.5)
ax.set_ylim(0.40, 1.00)
ax.set_xlabel('↑ higher = better', fontsize=8, color='dimgray')
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('comparison_accuracy_f1.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved → comparison_accuracy_f1.png')

## Cell 7 — Per-Class F1 Grouped Bar Chart

In [ ]:
class_labels = ['Low\n(0–280 kW)', 'Medium\n(280–830 kW)',
                'High\n(830–1165 kW)', 'Peak\n(1165+ kW)']
f1_cols = ['F1_Low', 'F1_Medium', 'F1_High', 'F1_Peak']
f1_data = cls_df[f1_cols].values     # shape (7, 4)

n_models  = len(models_cls)
n_classes = 4
x2        = np.arange(n_classes)
w2        = 0.10
half      = (n_models - 1) / 2
offsets   = [(i - half) * w2 for i in range(n_models)]

fig, ax = plt.subplots(figsize=(13, 6))
for i, (mname, color) in enumerate(zip(models_cls, colors_cls)):
    bs = ax.bar(x2 + offsets[i], f1_data[i], width=w2,
                color=color, edgecolor='white', linewidth=0.4,
                label=mname, alpha=0.90)
    for bar, val in zip(bs, f1_data[i]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=6)

ax.set_title('Per-Class F1-Score — All Models (5-Fold CV)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=10)
ax.set_xlabel('Load Class', fontsize=10)
ax.set_xticks(x2)
ax.set_xticklabels(class_labels, fontsize=10)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=8.5, loc='lower right', ncol=2)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('comparison_f1_per_class.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved → comparison_f1_per_class.png')

## Cell 8 — Overall Score Comparison — Radar Chart

In [ ]:
# Normalise each dimension 0–1 (higher = better for ALL axes)
# For error metrics (MAE, RMSE, MAPE): invert so smaller error = higher score
radar_data = pd.DataFrame({
    'Model':     reg_df['Model'].tolist(),
    'R²':        reg_df['R²'].values,
    '1-MAE_n':   1 - (reg_df['MAE'].values  / reg_df['MAE'].max()),
    '1-RMSE_n':  1 - (reg_df['RMSE'].values / reg_df['RMSE'].max()),
    '1-MAPE_n':  1 - (reg_df['MAPE (%)'].values / reg_df['MAPE (%)'].max()),
    'Accuracy':  cls_df['Accuracy (%)'].values / 100,
    'F1':        cls_df['F1 (weighted)'].values,
})

categories  = ['R²', 'Low MAE', 'Low RMSE', 'Low MAPE', 'Accuracy', 'F1-Score']
score_cols  = ['R²', '1-MAE_n', '1-RMSE_n', '1-MAPE_n', 'Accuracy', 'F1']
N           = len(categories)
angles      = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles     += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for _, row in radar_data.iterrows():
    model  = row['Model']
    values = [row[c] for c in score_cols] + [row[score_cols[0]]]
    color  = MODEL_COLORS[model]
    ax.plot(angles, values, color=color, linewidth=2, label=model)
    ax.fill(angles, values, color=color, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=7, color='grey')
ax.set_title('Overall Score Comparison — All Models (5-Fold CV)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.savefig('comparison_radar.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved → comparison_radar.png')

## Cell 9 — Combined Summary Heatmap

In [ ]:
# Build a unified summary table
summary = pd.DataFrame({
    'Model':       reg_df['Model'],
    'MAE':         reg_df['MAE'],
    'MSE':         reg_df['MSE'],
    'RMSE':        reg_df['RMSE'],
    'MAPE (%)':    reg_df['MAPE (%)'],
    'R²':          reg_df['R²'],
    'Accuracy (%)':cls_df['Accuracy (%)'],
    'F1 (wtd)':    cls_df['F1 (weighted)'],
    'F1-Low':      cls_df['F1_Low'],
    'F1-Med':      cls_df['F1_Medium'],
    'F1-High':     cls_df['F1_High'],
    'F1-Peak':     cls_df['F1_Peak'],
})
summary = summary.set_index('Model')

# Normalise 0–1 for colour coding (higher = better always after inversion)
lower_better = ['MAE', 'MSE', 'RMSE', 'MAPE (%)']
norm_df = summary.copy()
for col in norm_df.columns:
    mn, mx = norm_df[col].min(), norm_df[col].max()
    if mx == mn:
        norm_df[col] = 0.5
    elif col in lower_better:
        norm_df[col] = 1 - (norm_df[col] - mn) / (mx - mn)
    else:
        norm_df[col] = (norm_df[col] - mn) / (mx - mn)

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(norm_df.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(summary.columns)))
ax.set_xticklabels(summary.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(summary.index)))
ax.set_yticklabels(summary.index, fontsize=10)

# Annotate with actual values
for i in range(len(summary.index)):
    for j, col in enumerate(summary.columns):
        val  = summary.iloc[i, j]
        norm = norm_df.iloc[i, j]
        txt_color = 'black' if 0.25 < norm < 0.75 else ('white' if norm < 0.3 else 'black')
        if col in ['MAE', 'RMSE', 'MSE']:
            fmt = f'{val:.1f}'
        elif col in ['MAPE (%)', 'Accuracy (%)']:
            fmt = f'{val:.2f}'
        else:
            fmt = f'{val:.4f}'
        ax.text(j, i, fmt, ha='center', va='center',
                fontsize=7.5, color='black', fontweight='bold')

ax.set_title(
    'All-Model Performance Heatmap  (green = better, red = worse per column)',
    fontsize=12, fontweight='bold', pad=12
)
plt.colorbar(im, ax=ax, shrink=0.6, label='Normalised score (higher = better)')
plt.tight_layout()
plt.savefig('comparison_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved → comparison_heatmap.png')

## Cell 10 — Download All Figures

In [ ]:
from google.colab import files

figures = [
    'comparison_regression_metrics.png',
    'comparison_accuracy_f1.png',
    'comparison_f1_per_class.png',
    'comparison_radar.png',
    'comparison_heatmap.png',
]

for f in figures:
    files.download(f)
    print(f'Downloaded → {f}')

print('\nAll downloads complete ✓')